# 01 Baseline Evaluation

Purpose: run the untuned `Qwen/Qwen2.5-1.5B-Instruct` baseline evaluation directly from the Drive-backed source tree, save runtime outputs to `json-ft-runs`, and mirror the small final artifacts into the Drive source tree for later pullback into the repo.

Precondition: run `00_colab_setup.ipynb` first.


In [ ]:
from pathlib import Path

SOURCE_ROOT = Path('/content/drive/MyDrive/json-ft-source')
RUNTIME_ROOT = Path('/content/drive/MyDrive/json-ft-runs')
run_name = 'baseline-qwen2.5-1.5b'

print(f'Source root: {SOURCE_ROOT}')
print(f'Runtime root: {RUNTIME_ROOT}')


In [ ]:
!python /content/drive/MyDrive/json-ft-source/scripts/eval_model.py \
    --config /content/drive/MyDrive/json-ft-source/configs/eval.yaml \
    --run-name baseline-qwen2.5-1.5b \
    --runtime-root /content/drive/MyDrive/json-ft-runs \
    --dataset-path /content/drive/MyDrive/json-ft-source/data/manifests/support_tickets_eval_manifest.jsonl \
    --mirror-metrics-to-repo \
    --mirror-report-to-repo \
    --mirror-predictions-to-repo


In [ ]:
import json
from IPython.display import Markdown, display

metrics_path = SOURCE_ROOT / 'artifacts' / 'metrics' / f'{run_name}_metrics.json'
report_path = SOURCE_ROOT / 'artifacts' / 'reports' / f'{run_name}_report.md'
predictions_path = SOURCE_ROOT / 'artifacts' / 'reports' / f'{run_name}_predictions.jsonl'

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
print(f'Metrics path: {metrics_path}')
print(f'Report path: {report_path}')
print(f'Predictions path: {predictions_path}')

headline = '\n'.join([
    f"# Baseline Metrics: {run_name}",
    '',
    f"- JSON validity rate: `{metrics['json_validity_rate']:.4f}`",
    f"- Schema validation pass rate: `{metrics['schema_validation_pass_rate']:.4f}`",
    f"- Hallucinated field rate: `{metrics['hallucinated_field_rate']:.4f}`",
    f"- Field-level micro F1: `{metrics['field_level']['micro']['f1']:.4f}`",
    f"- Mean latency (ms): `{metrics['latency_ms']['mean']:.2f}`",
])
display(Markdown(headline))


In [ ]:
prediction_rows = [json.loads(line) for line in predictions_path.read_text(encoding='utf-8').splitlines() if line.strip()]
failures = [
    row
    for row in prediction_rows
    if row['parse_error'] or not row['schema_is_valid'] or row['parsed_payload'] != row['reference_payload']
]

print(f'Total prediction rows: {len(prediction_rows)}')
print(f'Failure rows: {len(failures)}')
for row in failures[:3]:
    print('\n' + '=' * 80)
    print(f"Record: {row['record_id']}")
    print(f"Parse error: {row['parse_error']}")
    print(f"Schema valid: {row['schema_is_valid']}")
    print(f"Unexpected fields: {row['unexpected_fields']}")
    print('Input text:')
    print(row['input_text'])
    print('Raw output:')
    print(row['raw_output'])
    print('Parsed payload:')
    print(json.dumps(row['parsed_payload'], indent=2, sort_keys=True) if row['parsed_payload'] is not None else 'null')
    print('Reference payload:')
    print(json.dumps(row['reference_payload'], indent=2, sort_keys=True))


In [ ]:
display(Markdown(report_path.read_text(encoding='utf-8')))
